# Tutorial 6: Real-World Application - Modeling Human Decision Making

This tutorial brings together everything we've covered by modeling a realistic cognitive task: multi-attribute decision making. We'll build a model of how people choose between apartments with different prices, sizes, commute distances, and amenities, then fit the model to behavioral data to capture individual differences in decision strategies.

## Why Model Real-World Tasks?

Real cognition rarely involves single, isolated processes. When making decisions, people juggle multiple strategies, work under time pressure, and bring their own preferences and abilities to the task. By modeling a complete task from start to finish, we can understand how different cognitive components interact and make testable predictions about human performance.

In [ ]:
import pyactr as actr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import time
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("Libraries loaded successfully!")

## 1. The Task: Apartment Choice

We'll model how people choose between apartments based on multiple attributes: price, size, commute distance, and amenities. This mirrors real consumer decisions where no single option dominates on all dimensions, forcing people to make trade-offs. The same framework applies to medical treatment choices, educational decisions, product selection, and more.

In [ ]:
def generate_apartments(n=6):
    """Generate realistic apartment options"""
    np.random.seed(42)
    
    apartments = []
    for i in range(n):
        base_quality = np.random.uniform(0.3, 0.9)
        
        apartment = {
            'id': chr(65 + i),
            'price': int(800 + (1 - base_quality) * 1200 + np.random.normal(0, 100)),
            'size': int(400 + base_quality * 600 + np.random.normal(0, 50)),
            'distance': round(2 + (1 - base_quality) * 8 + np.random.normal(0, 1), 1),
            'amenities': int(3 + base_quality * 6 + np.random.normal(0, 0.5))
        }
        
        apartment['amenities'] = max(1, min(10, apartment['amenities']))
        apartment['distance'] = max(0.5, apartment['distance'])
        
        apartments.append(apartment)
    
    return apartments

apartments = generate_apartments()
df_apartments = pd.DataFrame(apartments)

print("Apartment Options:\n")
print(df_apartments.to_string(index=False))

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 8))

ax1.scatter(df_apartments['price'], df_apartments['size'], s=100)
for i, apt in df_apartments.iterrows():
    ax1.annotate(apt['id'], (apt['price'], apt['size']), fontsize=12, ha='center')
ax1.set_xlabel('Price ($)')
ax1.set_ylabel('Size (sq ft)')
ax1.set_title('Price vs Size Trade-off')

ax2.scatter(df_apartments['price'], df_apartments['distance'], s=100)
for i, apt in df_apartments.iterrows():
    ax2.annotate(apt['id'], (apt['price'], apt['distance']), fontsize=12, ha='center')
ax2.set_xlabel('Price ($)')
ax2.set_ylabel('Distance (miles)')
ax2.set_title('Price vs Commute Trade-off')

ax3.scatter(df_apartments['size'], df_apartments['amenities'], s=100)
for i, apt in df_apartments.iterrows():
    ax3.annotate(apt['id'], (apt['size'], apt['amenities']), fontsize=12, ha='center')
ax3.set_xlabel('Size (sq ft)')
ax3.set_ylabel('Amenities Score')
ax3.set_title('Size vs Amenities')

colors = df_apartments['amenities']
sizes = 200 * (df_apartments['size'] / df_apartments['size'].max())
scatter = ax4.scatter(df_apartments['price'], df_apartments['distance'], 
                     c=colors, s=sizes, alpha=0.6, cmap='viridis')
for i, apt in df_apartments.iterrows():
    ax4.annotate(apt['id'], (apt['price'], apt['distance']), fontsize=12, ha='center')
ax4.set_xlabel('Price ($)')
ax4.set_ylabel('Distance (miles)')
ax4.set_title('All Attributes (size=bubble, color=amenities)')
plt.colorbar(scatter, ax=ax4, label='Amenities')

plt.tight_layout()
plt.show()

## 2. Cognitive Strategies for Decision Making

People use different strategies to make multi-attribute decisions. Weighted additive strategies compute a score for each option by summing weighted attributes. Elimination by aspects filters out options that don't meet thresholds on important attributes. Satisficing stops at the first acceptable option. Lexicographic strategies focus entirely on the most important attribute. The strategy someone uses depends on time pressure, cognitive load, and individual preferences.

In [ ]:
class DecisionModel:
    """ACT-R-inspired model of multi-attribute decision making"""
    
    def __init__(self, strategy="weighted", 
                 weights=None,
                 noise=0.5,
                 threshold=-2.0):
        
        self.strategy = strategy
        self.weights = weights or {'price': 0.4, 'size': 0.3, 'distance': 0.2, 'amenities': 0.1}
        
        self.model = actr.ACTRModel()
        self.model.model_parameters["subsymbolic"] = True
        
        actr.chunktype("apartment", "id price size distance amenities")
        actr.chunktype("goal_chunk", "state current_best current_score")
        actr.chunktype("comparison", "apt1 apt2 better")
        
        self.noise_level = noise
        self.decision_times = []
        self.comparisons_made = []
        
    def add_apartments_to_memory(self, apartments):
        """Add apartment options to declarative memory"""
        dm = self.model.decmem
        
        for apt in apartments:
            chunk = actr.makechunk(
                nameofchunk=f"apt_{apt['id']}",
                chunktype="apartment",
                id=apt['id'],
                price=str(apt['price']),
                size=str(apt['size']),
                distance=str(apt['distance']),
                amenities=str(apt['amenities'])
            )
            dm.add(chunk)
            
    def normalize_value(self, value, attribute, apartments_df):
        """Normalize attribute values to 0-1 scale"""
        min_val = apartments_df[attribute].min()
        max_val = apartments_df[attribute].max()
        
        if attribute in ['price', 'distance']:
            return (max_val - value) / (max_val - min_val)
        else:
            return (value - min_val) / (max_val - min_val)
    
    def weighted_additive_strategy(self, apartments):
        """WADD strategy: Calculate weighted sum for each option"""
        df = pd.DataFrame(apartments)
        scores = {}
        
        start_time = time.time()
        
        for apt in apartments:
            score = 0
            for attr in ['price', 'size', 'distance', 'amenities']:
                norm_value = self.normalize_value(apt[attr], attr, df)
                score += self.weights[attr] * norm_value
                time.sleep(0.05)
                
            scores[apt['id']] = score
            
        noise_factor = self.noise_level * 0.1
        for apt_id in scores:
            scores[apt_id] += np.random.normal(0, noise_factor)
            
        best_apt = max(scores, key=scores.get)
        decision_time = (time.time() - start_time) * 1000
        
        return best_apt, scores, decision_time
    
    def elimination_by_aspects_strategy(self, apartments, 
                                      thresholds=None):
        """EBA strategy: Eliminate options not meeting thresholds"""
        if thresholds is None:
            thresholds = {
                'price': 1500,
                'size': 600,
                'distance': 5,
                'amenities': 5
            }
            
        start_time = time.time()
        remaining = apartments.copy()
        
        attr_order = sorted(self.weights.keys(), 
                           key=lambda x: self.weights[x], 
                           reverse=True)
        
        for attr in attr_order:
            if len(remaining) == 1:
                break
                
            if attr in ['price', 'distance']:
                remaining = [apt for apt in remaining 
                           if apt[attr] <= thresholds[attr]]
            else:
                remaining = [apt for apt in remaining 
                           if apt[attr] >= thresholds[attr]]
                           
            time.sleep(0.1)
            
        if len(remaining) > 1:
            best, _, _ = self.weighted_additive_strategy(remaining)
            return best, {}, (time.time() - start_time) * 1000
        elif len(remaining) == 1:
            return remaining[0]['id'], {}, (time.time() - start_time) * 1000
        else:
            return apartments[0]['id'], {}, (time.time() - start_time) * 1000
    
    def make_decision(self, apartments):
        """Make decision using selected strategy"""
        self.add_apartments_to_memory(apartments)
        
        if self.strategy == "weighted":
            return self.weighted_additive_strategy(apartments)
        elif self.strategy == "eba":
            return self.elimination_by_aspects_strategy(apartments)

print("Testing Decision Strategies:\n")

model_wadd = DecisionModel(strategy="weighted", noise=0.3)
choice, scores, time_ms = model_wadd.make_decision(apartments)

print("1. Weighted Additive Strategy:")
print(f"   Choice: Apartment {choice}")
print(f"   Decision time: {time_ms:.0f}ms")
print("   Scores:")
for apt_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"     {apt_id}: {score:.3f}")

model_eba = DecisionModel(strategy="eba")
choice_eba, _, time_eba = model_eba.make_decision(apartments)

print(f"\n2. Elimination by Aspects Strategy:")
print(f"   Choice: Apartment {choice_eba}")
print(f"   Decision time: {time_eba:.0f}ms")

## 3. Fitting Models to Human Data

To test whether our model captures real human decision making, we need to fit it to behavioral data. We'll simulate human participants with individual differences in attribute weights, then recover those parameters using maximum likelihood estimation.

In [ ]:
def simulate_human_data(n_participants=30, n_trials=10):
    """Simulate human decision data with individual differences"""
    human_data = []
    
    for p in range(n_participants):
        weights = np.random.dirichlet([2, 2, 2, 2])
        participant_weights = {
            'price': weights[0],
            'size': weights[1],
            'distance': weights[2],
            'amenities': weights[3]
        }
        
        if p < n_participants * 0.7:
            strategy = "weighted"
        else:
            strategy = "eba"
            
        model = DecisionModel(strategy=strategy, 
                            weights=participant_weights,
                            noise=0.3 + np.random.normal(0, 0.1))
        
        for trial in range(n_trials):
            trial_apartments = generate_apartments()
            choice, _, decision_time = model.make_decision(trial_apartments)
            
            human_data.append({
                'participant': p,
                'trial': trial,
                'choice': choice,
                'rt': decision_time + np.random.normal(0, 50),
                'true_weights': participant_weights,
                'strategy': strategy
            })
            
    return pd.DataFrame(human_data)

human_df = simulate_human_data()

print("Simulated Human Data Summary:\n")
print(f"Participants: {human_df['participant'].nunique()}")
print(f"Trials per participant: {human_df['trial'].nunique()}")
print(f"Mean RT: {human_df['rt'].mean():.0f}ms (SD: {human_df['rt'].std():.0f}ms)")
print(f"\nChoice distribution:")
print(human_df['choice'].value_counts().sort_index())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

participant_rts = human_df.groupby('participant')['rt'].mean()
ax1.hist(participant_rts, bins=15, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Mean RT (ms)')
ax1.set_ylabel('Number of Participants')
ax1.set_title('Individual Differences in Decision Speed')

choice_matrix = pd.crosstab(human_df['participant'], human_df['choice'])
choice_diversity = choice_matrix.apply(lambda x: (x > 0).sum(), axis=1)
ax2.hist(choice_diversity, bins=range(1, 8), alpha=0.7, edgecolor='black')
ax2.set_xlabel('Number of Different Choices Made')
ax2.set_ylabel('Number of Participants')
ax2.set_title('Choice Diversity Across Trials')

plt.tight_layout()
plt.show()

## 4. Parameter Estimation and Model Fitting

Now we fit model parameters to individual participants using maximum likelihood estimation. This lets us recover each person's attribute weights and decision noise from their choice patterns.

In [ ]:
class ModelFitter:
    """Fit ACT-R model parameters to human data"""
    
    def __init__(self, human_data):
        self.human_data = human_data
        
    def likelihood(self, params, participant_data):
        """Calculate likelihood of parameters given participant's choices"""
        w_price, w_size, w_distance = params[:3]
        w_amenities = 1 - (w_price + w_size + w_distance)
        noise = params[3]
        
        if w_amenities < 0 or any(p < 0 or p > 1 for p in params[:3]):
            return -np.inf
            
        weights = {
            'price': w_price,
            'size': w_size,
            'distance': w_distance,
            'amenities': w_amenities
        }
        
        model = DecisionModel(weights=weights, noise=noise)
        log_likelihood = 0
        
        for _, row in participant_data.iterrows():
            np.random.seed(int(row['participant'] * 100 + row['trial']))
            trial_apartments = generate_apartments()
            choice, scores, _ = model.make_decision(trial_apartments)
            
            if scores:
                exp_scores = {k: np.exp(v * 5) for k, v in scores.items()}
                sum_exp = sum(exp_scores.values())
                probs = {k: v / sum_exp for k, v in exp_scores.items()}
                
                observed_choice = row['choice']
                if observed_choice in probs:
                    log_likelihood += np.log(probs[observed_choice] + 1e-10)
                else:
                    log_likelihood += np.log(1e-10)
                    
        return log_likelihood
    
    def fit_participant(self, participant_id):
        """Fit model to individual participant"""
        participant_data = self.human_data[self.human_data['participant'] == participant_id]
        
        initial_params = [0.25, 0.25, 0.25, 0.5]
        
        result = minimize(
            lambda p: -self.likelihood(p, participant_data),
            initial_params,
            method='Nelder-Mead',
            options={'maxiter': 100}
        )
        
        w_price, w_size, w_distance = result.x[:3]
        w_amenities = 1 - (w_price + w_size + w_distance)
        noise = result.x[3]
        
        return {
            'participant': participant_id,
            'fitted_w_price': w_price,
            'fitted_w_size': w_size,
            'fitted_w_distance': w_distance,
            'fitted_w_amenities': w_amenities,
            'fitted_noise': noise,
            'log_likelihood': -result.fun
        }

print("Fitting models to human data...\n")

fitter = ModelFitter(human_df)
fitted_params = []

for p in range(5):
    print(f"Fitting participant {p}...", end="")
    fit_result = fitter.fit_participant(p)
    fitted_params.append(fit_result)
    print(" Done!")

fitted_df = pd.DataFrame(fitted_params)

print("\nParameter Recovery (first participant):")
p0_true = human_df[human_df['participant'] == 0].iloc[0]['true_weights']
p0_fitted = fitted_df[fitted_df['participant'] == 0].iloc[0]

print(f"\n{'Attribute':<12} {'True Weight':<12} {'Fitted Weight':<12} {'Error':<12}")
print("-" * 48)
for attr in ['price', 'size', 'distance', 'amenities']:
    true_w = p0_true[attr]
    fitted_w = p0_fitted[f'fitted_w_{attr}']
    error = fitted_w - true_w
    print(f"{attr:<12} {true_w:<12.3f} {fitted_w:<12.3f} {error:+12.3f}")

## 5. Applications to Real-World Problems

### 5.1 Human-Computer Interaction (HCI)

Cognitive models can predict how interface design affects decision quality and speed. By testing different information displays in simulation, we can optimize designs before running expensive user studies.

In [ ]:
class InterfaceDesignModel:
    """Model how interface design affects decision making"""
    
    def __init__(self):
        self.base_model = DecisionModel()
        
    def test_information_display(self, apartments, display_format):
        """Test how different displays affect decisions"""
        
        if display_format == "table":
            processing_time = 50
            error_rate = 0.02
            
        elif display_format == "sequential":
            processing_time = 100
            error_rate = 0.05
            
        elif display_format == "graphical":
            processing_time = 30
            error_rate = 0.03
            
        start_time = time.time()
        n_comparisons = len(apartments) * 4
        total_time = n_comparisons * processing_time
        
        if np.random.random() < error_rate:
            choice = np.random.choice([apt['id'] for apt in apartments])
        else:
            choice, _, _ = self.base_model.make_decision(apartments)
            
        return {
            'format': display_format,
            'choice': choice,
            'time': total_time,
            'error': np.random.random() < error_rate
        }

print("Interface Design Analysis:\n")

interface_model = InterfaceDesignModel()
formats = ["table", "sequential", "graphical"]
results = []

for format_type in formats:
    format_results = []
    for _ in range(50):
        result = interface_model.test_information_display(apartments, format_type)
        format_results.append(result)
    results.extend(format_results)

results_df = pd.DataFrame(results)

summary = results_df.groupby('format').agg({
    'time': 'mean',
    'error': 'mean',
    'choice': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
})

print(summary)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

formats_ordered = ['graphical', 'table', 'sequential']
times = [summary.loc[f, 'time'] for f in formats_ordered]
ax1.bar(formats_ordered, times)
ax1.set_ylabel('Decision Time (ms)')
ax1.set_title('Speed by Display Format')

errors = [summary.loc[f, 'error'] * 100 for f in formats_ordered]
ax2.bar(formats_ordered, errors)
ax2.set_ylabel('Error Rate (%)')
ax2.set_title('Accuracy by Display Format')

plt.tight_layout()
plt.show()

print("\nDesign Recommendations:")
print("- Graphical displays: Fastest for comparisons")
print("- Table format: Good balance of speed and accuracy")
print("- Sequential display: Avoid for complex decisions")

### 5.2 Educational Applications

Modeling skill acquisition helps us understand how decision-making expertise develops and how to design effective training programs.

In [ ]:
class LearningModel:
    """Model skill acquisition in decision making"""
    
    def __init__(self):
        self.experience = 0
        self.skill_level = 0.0
        
    def practice_trial(self, apartments, optimal_choice):
        """Simulate one practice trial with feedback"""
        
        if self.skill_level < 0.3:
            strategy = "random"
            choice = np.random.choice([apt['id'] for apt in apartments])
            time = 2000
            
        elif self.skill_level < 0.7:
            model = DecisionModel(strategy="weighted", noise=1.0)
            choice, _, _ = model.make_decision(apartments)
            time = 1500
            
        else:
            model = DecisionModel(strategy="weighted", noise=0.3)
            choice, _, _ = model.make_decision(apartments)
            time = 800
            
        correct = (choice == optimal_choice)
        if correct:
            self.skill_level += 0.1 * (1 - self.skill_level)
        else:
            self.skill_level += 0.02
            
        self.experience += 1
        
        return {
            'trial': self.experience,
            'choice': choice,
            'correct': correct,
            'time': time,
            'skill': self.skill_level
        }

print("Modeling Learning Curve:\n")

learner = LearningModel()
learning_data = []

for trial in range(30):
    trial_apts = generate_apartments()
    model = DecisionModel()
    optimal, _, _ = model.make_decision(trial_apts)
    result = learner.practice_trial(trial_apts, optimal)
    learning_data.append(result)

learning_df = pd.DataFrame(learning_data)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))

ax1.plot(learning_df['trial'], learning_df['skill'], 'b-', linewidth=2)
ax1.set_xlabel('Trial')
ax1.set_ylabel('Skill Level')
ax1.set_title('Skill Acquisition Curve')
ax1.grid(True, alpha=0.3)

window = 5
accuracy = learning_df['correct'].rolling(window).mean()
ax2.plot(learning_df['trial'], accuracy, 'g-', linewidth=2)
ax2.set_xlabel('Trial')
ax2.set_ylabel(f'Accuracy ({window}-trial average)')
ax2.set_title('Decision Accuracy Improvement')
ax2.grid(True, alpha=0.3)

ax3.plot(learning_df['trial'], learning_df['time'], 'r-', linewidth=2)
ax3.set_xlabel('Trial')
ax3.set_ylabel('Decision Time (ms)')
ax3.set_title('Speed Improvement')
ax3.grid(True, alpha=0.3)

trials = np.arange(1, 31)
theoretical_time = 2000 * np.power(trials, -0.3)
ax4.loglog(learning_df['trial'], learning_df['time'], 'ro', label='Observed')
ax4.loglog(trials, theoretical_time, 'b-', label='Power Law')
ax4.set_xlabel('Trial (log scale)')
ax4.set_ylabel('Time (log scale)')
ax4.set_title('Power Law of Practice')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nEducational Insights:")
print(f"- Initial accuracy: {learning_df.iloc[:5]['correct'].mean():.0%}")
print(f"- Final accuracy: {learning_df.iloc[-5:]['correct'].mean():.0%}")
print(f"- Speed improvement: {learning_df.iloc[0]['time']/learning_df.iloc[-1]['time']:.1f}x")
print("\nInstructional recommendations:")
print("- Provide immediate feedback")
print("- Start with simple comparisons")
print("- Gradually increase complexity")
print("- Focus on strategy development")

## 6. Individual Differences and Personalization

Real applications must account for individual differences in cognitive capacity, processing speed, and risk preferences. These parameters create distinct behavioral patterns that can inform personalized system design.

In [ ]:
class PersonalizedModel:
    """Model with individual difference parameters"""
    
    def __init__(self, cognitive_profile):
        self.working_memory = cognitive_profile.get('working_memory', 1.0)
        self.processing_speed = cognitive_profile.get('processing_speed', 1.0)
        self.risk_preference = cognitive_profile.get('risk_preference', 0.5)
        
        self.model = actr.ACTRModel()
        self.model.model_parameters["ans"] = 0.5 / self.working_memory
        self.model.model_parameters["latency_factor"] = 0.2 / self.processing_speed
        
    def predict_choice_pattern(self, n_trials=20):
        """Predict individual's choice patterns"""
        choices = []
        
        for _ in range(n_trials):
            apartments = generate_apartments()
            
            if self.risk_preference < 0.3:
                weights = {'price': 0.5, 'size': 0.2, 'distance': 0.2, 'amenities': 0.1}
            elif self.risk_preference > 0.7:
                weights = {'price': 0.2, 'size': 0.3, 'distance': 0.1, 'amenities': 0.4}
            else:
                weights = {'price': 0.3, 'size': 0.3, 'distance': 0.2, 'amenities': 0.2}
                
            model = DecisionModel(weights=weights, 
                                noise=0.5 / self.working_memory)
            choice, _, _ = model.make_decision(apartments)
            choices.append(choice)
            
        return choices

print("Individual Difference Modeling:\n")

profiles = {
    "High Capacity": {
        "working_memory": 1.5,
        "processing_speed": 1.5,
        "risk_preference": 0.5
    },
    "Low Capacity": {
        "working_memory": 0.5,
        "processing_speed": 0.7,
        "risk_preference": 0.3
    },
    "Risk Seeker": {
        "working_memory": 1.0,
        "processing_speed": 1.0,
        "risk_preference": 0.8
    }
}

profile_predictions = {}

for profile_name, profile in profiles.items():
    model = PersonalizedModel(profile)
    choices = model.predict_choice_pattern()
    
    profile_predictions[profile_name] = {
        "choices": choices,
        "diversity": len(set(choices)),
        "most_common": max(set(choices), key=choices.count)
    }
    
    print(f"\n{profile_name}:")
    print(f"  Choice diversity: {profile_predictions[profile_name]['diversity']} different options")
    print(f"  Most frequent choice: Apartment {profile_predictions[profile_name]['most_common']}")
    print(f"  Profile: WM={profile['working_memory']}, Speed={profile['processing_speed']}, Risk={profile['risk_preference']}")

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for name, profile in profiles.items():
    ax.scatter(profile['working_memory'], 
              profile['processing_speed'],
              s=200 * profile['risk_preference'],
              alpha=0.6,
              label=name)
    
ax.set_xlabel('Working Memory Capacity')
ax.set_ylabel('Processing Speed')
ax.set_title('Individual Cognitive Profiles\n(bubble size = risk preference)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nPersonalization Applications:")
print("- Adaptive interfaces that match cognitive capacity")
print("- Personalized decision support systems")
print("- Tailored educational interventions")
print("- Individual difference assessment tools")

## Exercises

1. Adapt the apartment choice model to a different domain like job selection or medical treatment choice. What attributes matter? How might strategies differ?

2. Modify the model to include time pressure. How do strategies change when decisions must be made quickly?

3. Extend the model to simulate group decision making. How do multiple agents with different preferences reach consensus?

4. Model how expertise in one domain transfers to another. Does someone skilled at apartment hunting make better car-buying decisions?

## Further Reading

Payne, J. W., Bettman, J. R., & Johnson, E. J. (1993). The Adaptive Decision Maker. Cambridge University Press.

Salvucci, D. D. (2006). Modeling driver behavior in a cognitive architecture. Human Factors, 48(2), 362-380.

Anderson, J. R., Bothell, D., Byrne, M. D., Douglass, S., Lebiere, C., & Qin, Y. (2004). An integrated theory of the mind. Psychological Review, 111(4), 1036-1060.

---

This tutorial demonstrated how ACT-R models bridge cognitive theory, behavioral data, and practical applications. The models we built can predict human performance, capture individual differences, and inform the design of better systems. With these skills, you can model your own tasks, contribute to cognitive science research, and apply computational insights to real-world problems.